IV.a) Match the measured PFC miRNAs to modern identifiers and TargetScan seed families for the miRDB analysis.

This notebook has one purpose: to create pfc_mirna_annotation.csv, the miRNA-to-seed annotation used as input in notebook 3c. It does not use miRTarBase or TargetScan target predictions.

The matching chain is:

old chip identifier → miRBase 12 accession → TargetScan 8.0 miRNA identifier and Seed+m8

1.) Import libraries

In [1]:
import pandas as pd
from IPython.display import display

2.) Load the measured PFC miRNA expression data.

The row index contains the original miRNA identifiers from the processed microarray data. Only these measured miRNAs are annotated.

In [2]:
pfc_mirna = pd.read_csv(
    "results/pfc_mirna_expression.csv",
    index_col=0
)

print("Measured PFC miRNAs:", pfc_mirna.index.nunique())
display(pfc_mirna.head())

Measured PFC miRNAs: 198


,Hsa_PFC_3days,Hsa_PFC_5days,Hsa_PFC_95days,Hsa_PFC_788days,Hsa_PFC_1631days,Hsa_PFC_9278days,Hsa_PFC_24091days,Hsa_PFC_32121days,Ptr_PFC_1days,Ptr_PFC_2days,...,Ptr_PFC_46days,Ptr_PFC_4416days,Mml_PFC_2days,Mml_PFC_8days,Mml_PFC_17days,Mml_PFC_450days,Mml_PFC_1488days,Mml_PFC_3390days,Mml_PFC_8105days,Mml_PFC_10221days
ID_REF,,,,,,,,,,,,,,,,,,,,,
hsa-let-7a,12.509508,12.509508,12.964535,12.706113,12.509508,13.416576,12.964535,13.416576,12.950516,12.761680,...,12.863056,13.315097,12.509508,12.328442,12.328442,12.328442,12.509508,12.509508,12.509508,12.706113
hsa-let-7b,12.328442,12.328442,12.328442,12.173784,12.005529,12.509508,12.005529,12.328442,12.166797,12.250924,...,11.728212,12.404814,11.651839,11.844948,11.651839,11.844948,12.005529,11.844948,11.844948,11.651839
hsa-let-7b*,4.371074,3.705721,3.541066,3.290490,3.705721,3.172556,3.674399,4.087028,3.826922,4.114400,...,4.016234,3.353728,3.387815,4.087028,3.744282,3.487588,3.819235,3.387815,3.764886,3.487588
hsa-let-7c,11.474729,11.651839,11.474729,10.782649,10.782649,10.713628,10.612121,11.083687,11.279318,11.563276,...,10.478911,10.478911,11.474729,11.474729,11.474729,10.967484,10.967484,10.871652,10.871652,10.967484
hsa-let-7d,9.452227,9.575366,9.407652,9.575366,9.618224,9.783828,9.783828,9.919043,9.777656,9.777656,...,9.610065,9.510561,9.844862,9.989534,9.575366,9.783828,10.037649,10.138753,10.037649,10.037649


3.) Match the old chip identifiers to miRBase 12 from 2008.

The experiment used an older miRNA naming convention. miRBase 12 provides the contemporaneous name, the mature-miRNA sequence, and the stable MIMAT accession used to bridge to the newer annotation.

In [3]:
mirbase12 = pd.read_csv("data/miRBase12_human.csv")

mirna_annotation = pd.DataFrame({
    "miRNA_original": pfc_mirna.index.astype(str).str.strip()
})

mirna_annotation["match_name"] = (
    mirna_annotation["miRNA_original"]
    .str.lower()
)

mirbase12["match_name"] = (
    mirbase12["NAME"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mirbase12_mapping = (
    mirbase12[
        ["match_name", "ACCESSION", "SEQUENCE"]
    ]
    .drop_duplicates()
    .drop_duplicates(subset="match_name")
)

mirna_annotation = mirna_annotation.merge(
    mirbase12_mapping,
    on="match_name",
    how="left",
    validate="one_to_one"
)

print("Measured miRNAs:", len(mirna_annotation))
print(
    "Matched to miRBase 12:",
    mirna_annotation["ACCESSION"].notna().sum()
)
print(
    "Not matched to miRBase 12:",
    mirna_annotation["ACCESSION"].isna().sum()
)

display(mirna_annotation.head())

Measured miRNAs: 198
Matched to miRBase 12: 198
Not matched to miRBase 12: 0


,miRNA_original,match_name,ACCESSION,SEQUENCE
0,hsa-let-7a,hsa-let-7a,MIMAT0000062,UGAGGUAGUAGGUUGUAUAGUU
1,hsa-let-7b,hsa-let-7b,MIMAT0000063,UGAGGUAGUAGGUUGUGUGGUU
2,hsa-let-7b*,hsa-let-7b*,MIMAT0004482,CUAUACAACCUACUGCCUUCCC
3,hsa-let-7c,hsa-let-7c,MIMAT0000064,UGAGGUAGUAGGUUGUAUGGUU
4,hsa-let-7d,hsa-let-7d,MIMAT0000065,AGAGGUAGUAGGUUGCAUAGUU


4.) Load the TargetScan 8.0 miRNA-family annotation.

We use miR_Family_Info.txt only as an identifier and seed-annotation resource. TargetScan target predictions are not used in the miRDB network.

In [4]:
mirna_families = pd.read_csv(
    "data/miR_Family_Info.txt",
    sep="	"
)

mirna_families_human = mirna_families[
    mirna_families["Species ID"].astype(str) == "9606"
].copy()

print("Human TargetScan annotation rows:", len(mirna_families_human))
print(
    "Distinct TargetScan seed families:",
    mirna_families_human["Seed+m8"].nunique()
)

display(mirna_families_human.head())

Human TargetScan annotation rows: 2606
Distinct TargetScan seed families: 2064


,miR family,Seed+m8,Species ID,MiRBase ID,Mature sequence,Family Conservation?,MiRBase Accession
53,let-7-5p/98-5p,GAGGUAG,9606,hsa-let-7a-5p,UGAGGUAGUAGGUUGUAUAGUU,2,MIMAT0000062
54,let-7-5p/98-5p,GAGGUAG,9606,hsa-let-7b-5p,UGAGGUAGUAGGUUGUGUGGUU,2,MIMAT0000063
55,let-7-5p/98-5p,GAGGUAG,9606,hsa-let-7c-5p,UGAGGUAGUAGGUUGUAUGGUU,2,MIMAT0000064
56,let-7-5p/98-5p,GAGGUAG,9606,hsa-let-7d-5p,AGAGGUAGUAGGUUGCAUAGUU,2,MIMAT0000065
57,let-7-5p/98-5p,GAGGUAG,9606,hsa-let-7e-5p,UGAGGUAGGAGGUUGUAUAGUU,2,MIMAT0000066


5.) Match the MIMAT accessions to TargetScan 8.0 identifiers and seed families.

The stable accession links the old miRBase 12 name to the newer TargetScan annotation. Seed+m8 is the seed sequence used to group measured miRNAs into seed families in the miRDB analysis.

In [5]:
targetscan_annotation = (
    mirna_families_human[
        [
            "MiRBase Accession",
            "MiRBase ID",
            "miR family",
            "Seed+m8",
        ]
    ]
    .rename(columns={"MiRBase Accession": "ACCESSION"})
    .drop_duplicates()
    .drop_duplicates(subset="ACCESSION")
)

mirna_annotation = mirna_annotation.merge(
    targetscan_annotation,
    on="ACCESSION",
    how="left",
    validate="many_to_one"
)

print(
    "miRNAs with a TargetScan identifier:",
    mirna_annotation["MiRBase ID"].notna().sum()
)
print(
    "miRNAs with a seed annotation:",
    mirna_annotation["Seed+m8"].notna().sum()
)
print(
    "Distinct measured seed families:",
    mirna_annotation["Seed+m8"].nunique()
)

display(mirna_annotation.head(10))

miRNAs with a TargetScan identifier: 197
miRNAs with a seed annotation: 197
Distinct measured seed families: 157


,miRNA_original,match_name,ACCESSION,SEQUENCE,MiRBase ID,miR family,Seed+m8
0,hsa-let-7a,hsa-let-7a,MIMAT0000062,UGAGGUAGUAGGUUGUAUAGUU,hsa-let-7a-5p,let-7-5p/98-5p,GAGGUAG
1,hsa-let-7b,hsa-let-7b,MIMAT0000063,UGAGGUAGUAGGUUGUGUGGUU,hsa-let-7b-5p,let-7-5p/98-5p,GAGGUAG
2,hsa-let-7b*,hsa-let-7b*,MIMAT0004482,CUAUACAACCUACUGCCUUCCC,hsa-let-7b-3p,let-7a-3p/let-7b-3p/let-7f-1-3p/98-3p,UAUACAA
3,hsa-let-7c,hsa-let-7c,MIMAT0000064,UGAGGUAGUAGGUUGUAUGGUU,hsa-let-7c-5p,let-7-5p/98-5p,GAGGUAG
4,hsa-let-7d,hsa-let-7d,MIMAT0000065,AGAGGUAGUAGGUUGCAUAGUU,hsa-let-7d-5p,let-7-5p/98-5p,GAGGUAG
5,hsa-let-7e,hsa-let-7e,MIMAT0000066,UGAGGUAGGAGGUUGUAUAGUU,hsa-let-7e-5p,let-7-5p/98-5p,GAGGUAG
6,hsa-let-7f,hsa-let-7f,MIMAT0000067,UGAGGUAGUAGAUUGUAUAGUU,hsa-let-7f-5p,let-7-5p/98-5p,GAGGUAG
7,hsa-let-7g,hsa-let-7g,MIMAT0000414,UGAGGUAGUAGUUUGUACAGUU,hsa-let-7g-5p,let-7-5p/98-5p,GAGGUAG
8,hsa-let-7i,hsa-let-7i,MIMAT0000415,UGAGGUAGUAGUUUGUGCUGUU,hsa-let-7i-5p,let-7-5p/98-5p,GAGGUAG
9,hsa-mir-1,hsa-mir-1,MIMAT0000416,UGGAAUGUAAAGAAGUAUGUAU,hsa-miR-1-3p,miR-1-3p/206,GGAAUGU


6.) Check the completed annotation and save the output for notebook 4b.

The final file contains one row per measured PFC miRNA. Notebook 4bc uses MiRBase ID to align the measured miRNAs with miRDB names and uses Seed+m8 to construct the seed–gene network.

In [6]:
output_columns = [
    "miRNA_original",
    "match_name",
    "ACCESSION",
    "SEQUENCE",
    "MiRBase ID",
    "miR family",
    "Seed+m8",
]

mirna_annotation = mirna_annotation[output_columns].copy()

if mirna_annotation["miRNA_original"].duplicated().any():
    raise ValueError("Duplicate measured miRNA rows were created during matching.")

unmatched = mirna_annotation[
    mirna_annotation["Seed+m8"].isna()
].copy()

print("Final annotation rows:", len(mirna_annotation))
print("Measured seed families:", mirna_annotation["Seed+m8"].nunique())
print("miRNAs without a seed annotation:", len(unmatched))

if len(unmatched) > 0:
    display(
        unmatched[
            ["miRNA_original", "ACCESSION", "MiRBase ID"]
        ]
    )

mirna_annotation.to_csv(
    "results/pfc_mirna_annotation.csv",
    index=False
)

print("Saved: results/pfc_mirna_annotation.csv")
display(mirna_annotation.head())

Final annotation rows: 198
Measured seed families: 157
miRNAs without a seed annotation: 1


,miRNA_original,ACCESSION,MiRBase ID
186,hsa-mir-886-3p,MIMAT0004906,NaN


Saved: results/pfc_mirna_annotation.csv


,miRNA_original,match_name,ACCESSION,SEQUENCE,MiRBase ID,miR family,Seed+m8
0,hsa-let-7a,hsa-let-7a,MIMAT0000062,UGAGGUAGUAGGUUGUAUAGUU,hsa-let-7a-5p,let-7-5p/98-5p,GAGGUAG
1,hsa-let-7b,hsa-let-7b,MIMAT0000063,UGAGGUAGUAGGUUGUGUGGUU,hsa-let-7b-5p,let-7-5p/98-5p,GAGGUAG
2,hsa-let-7b*,hsa-let-7b*,MIMAT0004482,CUAUACAACCUACUGCCUUCCC,hsa-let-7b-3p,let-7a-3p/let-7b-3p/let-7f-1-3p/98-3p,UAUACAA
3,hsa-let-7c,hsa-let-7c,MIMAT0000064,UGAGGUAGUAGGUUGUAUGGUU,hsa-let-7c-5p,let-7-5p/98-5p,GAGGUAG
4,hsa-let-7d,hsa-let-7d,MIMAT0000065,AGAGGUAGUAGGUUGCAUAGUU,hsa-let-7d-5p,let-7-5p/98-5p,GAGGUAG


Result: results/pfc_mirna_annotation.csv is the only output from this notebook. It is the miRNA seed-annotation input required by 4b_miRNA_targets_matching_miRDB.